In [0]:
pip  install holidays

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import holidays
import requests 

In [0]:
taxi_zone_df = spark.read.csv("/Volumes/workspace/default/raw_data/taxi_zone_lookup.csv", header=True)

In [0]:
yellow_taxi_trip_df = spark.read.parquet("/Volumes/workspace/default/raw_data/yellow_tripdata_2023-01.parquet")

In [0]:
# Période à générer
start_date = "2026-01-01"
end_date = "2026-12-31"

# Création de la plage de dates
dates = pd.date_range(start=start_date, end=end_date, freq="D")

# Jours fériés France
fr_holidays = holidays.France(years=[2026])

# Création du DataFrame
df = pd.DataFrame({
    "date": dates,
})

df["day_of_week"] = df["date"].dt.day_name()
df["is_weekend"] = df["date"].dt.weekday >= 5
df["is_holiday"] = df["date"].dt.date.astype("datetime64[ns]").isin(pd.to_datetime(list(fr_holidays.keys())))

# Format date en texte
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

# Export CSV
calendar_df = spark.createDataFrame(df)

In [0]:
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude":"40.71",
    "longitude":"74.00",
    "hourly":["temperature_2m","precipitation","windspeed_10m"]
}
response = requests.get(url, params=params).json()
hourly = response["hourly"]
df = pd.DataFrame(hourly)
df["latitude"] = response["latitude"]
df["longitude"] = response["longitude"]
weather_df = spark.createDataFrame(df)

#weather_df = spark.read.json(weather)